# Kappa - Daughter-specific chromatin changes tuning

- Added DG1 bias handling with 1.4 weighting of L2 regularization of CG1/DG1 branches. With scaling curve wrt to kappa. see: dg1_bias_regularization.py


In [1]:
# Preamble, notebook setup and imports

%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
%config Completer.use_jedi = False

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [2]:
from src.utils import mkdir_safe
from src.chromatin_deconvolution_solver import plot_example_fits
from src.chromatin_deconvolution_solver import plot_branches
from src.chromatin_model import plot_prediction
from src.geneset import get_deconvolved_geneset
from src.sgd import get_orfname
from src.RealDataReplication import read_n_fr_b
from src.chromatin_deconvolution_solver import subset_select_highest_G_indices


In [3]:
from src.chromatin_model import ChromatinModel
from src.config import load_default_chrom_configs

config1, config2 = load_default_chrom_configs('distinct')
chromatin_model1 = ChromatinModel(config1)


todo: Loading testing config from replication deconvolution


In [871]:
from src.chromatin_gamma_search import ChromatinFindOptimalGamma
from src.chromatin_deconvolution_solver import ChromatinDeconvolveSolver

In [5]:
optimal_gamma = 0.00763959944903568 # from previous gamma sweep
# todo: notebook for gamma sweep, separate logic


In [6]:
genes = get_deconvolved_geneset()

In [9]:
intergenic_regions_df = pd.read_csv('datasets/computed_mnase/intergenic_regions_gene500_chrom1000_min500.csv',)
intergenic_regions_df.head()

,chromosome,start,end,length,coverage,mid,name
0,1,3207,6735,3528,0.999433,4971,1_4971
1,1,9516,12863,3347,0.893305,11189,1_11189
2,1,14243,21066,6823,0.965406,17654,1_17654
3,1,28712,30689,1977,0.999494,29700,1_29700
4,1,70082,71260,1178,0.999150,70671,1_70671


In [11]:
# Let's sweep these genes and examine manually
gene_names = ['DSE1', 'CLB2', 'MCM6']
intergenic_names = []

In [13]:
from src.find_optimal_kappa_chromatin import DSE_GENES, INTERGENICS
from src.find_optimal_kappa_chromatin import deconvolve_and_plot_region, \
    deconvolve_and_plot_gene
from src.timer import Timer

directory = f'output/chromatin_dg1_bias_correction/'
mkdir_safe(directory)

filename_format = "{}_kappa{}_gamma{}_F.npy"

gamma = 0.0076

num_kappas = 20
k_min, k_max = 1e-4, 1e-1
logks = np.linspace(np.log10(k_min), np.log10(k_max), num_kappas)
kappas = 10**logks

window = 500
win_2 = window//2
groups = ['gene', 'intergenic']
meta_df = pd.DataFrame()

timer = Timer()

from src.geneset import get_deconvolved_geneset
from src.sgd import get_orfname

genes = get_deconvolved_geneset()
    
for k_i, kappa in enumerate(kappas):
    print(f"[{k_i}] Kappa: ", kappa)

    for g_i, gene_name in enumerate(gene_names):
        
        orf_name = get_orfname(gene_name)
        if orf_name not in genes.index: 
            print(f"{gene_name} not in gene set. Skipping")
            continue
        
        gene = genes.loc[orf_name]
        
        filename = filename_format.format(gene_name, kappa, gamma)
        save_path = f"{directory}/{filename}"
            
        span = gene.TSS-win_2, gene.TSS+win_2

        row_df = pd.DataFrame({
            'path': save_path,
            'name': gene_name,
            'group': 'gene',
            'kappa': kappa,
            'gamma': gamma,
            'window': window,
            'chrom': gene.chr,
            'start': span[0],
            'end': span[1]
        }, index=[0])
        
        # Deconvolve and save to disk
        deconvolve_and_plot_gene(config1, gene_name, gamma=gamma, 
                                 kappa=kappa, 
            output_directory=None, f_savepath=save_path, window=500, 
                                 save_plots=False)

        timer.print_time(f"Saved: {save_path}")
        meta_df = pd.concat([meta_df, row_df])

    for i_i, interg_name in enumerate(intergenic_names):
        filename = filename_format.format(interg_name, kappa, gamma)
        interg_split_name = interg_name.split('_')
        chrom, mid = int(interg_split_name[0]), int(interg_split_name[1])
        save_path = f"{directory}/{filename}"
        span = mid-win_2, mid+win_2

        row_df = pd.DataFrame({
            'path': save_path,
            'name': interg_name,
            'group': 'intergenic',
            'kappa': kappa,
            'gamma': gamma,
            'window': window,
            'chrom': chrom,
            'start': span[0],
            'end': span[1]
        }, index=[0])

        # Deconvolve and save to disk
        deconvolve_and_plot_region(config1, chrom, mid, gamma=gamma, kappa=kappa, 
            output_directory=None, f_savepath=save_path, window=500, save_plots=False)
        timer.print_time(f"Saved: {save_path}")
        meta_df = pd.concat([meta_df, row_df])

meta_df = meta_df.set_index('name')
meta_df.to_csv(f'{directory}/meta.csv')


Creating directory: output/chromatin_dg1_bias_correction/...Directory exists. Skipping.
[0] Kappa:  0.0001
Loading trial replication profile, 2/17/25
Loading chromosome reads: 5
Applying a normalization for length distribution
0/1300 - 00:00:00.001
500/1300 - 00:00:45.442
1000/1300 - 00:01:55.116
Saved: output/chromatin_dg1_bias_correction//DSE1_kappa0.0001_gamma0.0076_F.npy - 00:02:42.692
Loading trial replication profile, 2/17/25
Loading chromosome reads: 16
Applying a normalization for length distribution
0/1300 - 00:00:00.000
500/1300 - 00:00:34.175
1000/1300 - 00:01:38.760
Saved: output/chromatin_dg1_bias_correction//CLB2_kappa0.0001_gamma0.0076_F.npy - 00:05:07.662
Loading trial replication profile, 2/17/25
Loading chromosome reads: 7
Applying a normalization for length distribution
0/1300 - 00:00:00.001
500/1300 - 00:00:28.146
1000/1300 - 00:01:28.513
Saved: output/chromatin_dg1_bias_correction//MCM6_kappa0.0001_gamma0.0076_F.npy - 00:07:22.136
[1] Kappa:  0.0001438449888287663


Loading chromosome reads: 7
Applying a normalization for length distribution
0/1300 - 00:00:00.000
500/1300 - 00:00:27.523
1000/1300 - 00:01:21.812
Saved: output/chromatin_dg1_bias_correction//MCM6_kappa0.0018329807108324356_gamma0.0076_F.npy - 00:58:09.945
[9] Kappa:  0.0026366508987303583
Loading trial replication profile, 2/17/25
Loading chromosome reads: 5
Applying a normalization for length distribution
0/1300 - 00:00:00.001
500/1300 - 00:00:28.374
1000/1300 - 00:01:27.001
Saved: output/chromatin_dg1_bias_correction//DSE1_kappa0.0026366508987303583_gamma0.0076_F.npy - 01:00:10.003
Loading trial replication profile, 2/17/25
Loading chromosome reads: 16
Applying a normalization for length distribution
0/1300 - 00:00:00.000
500/1300 - 00:00:30.737
1000/1300 - 00:01:33.794
Saved: output/chromatin_dg1_bias_correction//CLB2_kappa0.0026366508987303583_gamma0.0076_F.npy - 01:02:29.482
Loading trial replication profile, 2/17/25
Loading chromosome reads: 7
Applying a normalization for lengt

500/1300 - 00:00:27.164
1000/1300 - 00:01:23.164
Saved: output/chromatin_dg1_bias_correction//CLB2_kappa0.04832930238571752_gamma0.0076_F.npy - 01:50:07.409
Loading trial replication profile, 2/17/25
Loading chromosome reads: 7
Applying a normalization for length distribution
0/1300 - 00:00:00.000
500/1300 - 00:00:25.387
1000/1300 - 00:01:14.870
Saved: output/chromatin_dg1_bias_correction//MCM6_kappa0.04832930238571752_gamma0.0076_F.npy - 01:52:01.960
[18] Kappa:  0.06951927961775606
Loading trial replication profile, 2/17/25
Loading chromosome reads: 5
Applying a normalization for length distribution
0/1300 - 00:00:00.001
500/1300 - 00:00:23.006
1000/1300 - 00:01:09.724
Saved: output/chromatin_dg1_bias_correction//DSE1_kappa0.06951927961775606_gamma0.0076_F.npy - 01:53:39.666
Loading trial replication profile, 2/17/25
Loading chromosome reads: 16
Applying a normalization for length distribution
0/1300 - 00:00:00.000
500/1300 - 00:00:23.173
1000/1300 - 00:01:13.472
Saved: output/chroma

In [15]:
meta_df = pd.read_csv(f"{directory}/meta.csv").set_index(['name', 'kappa'])
meta_df.head()

,,path,group,gamma,window,chrom,start,end
name,kappa,,,,,,,
DSE1,0.000100,output/chromatin_dg1_bias_correction//DSE1_kap...,gene,0.0076,500,5,408854,409354
CLB2,0.000100,output/chromatin_dg1_bias_correction//CLB2_kap...,gene,0.0076,500,16,771055,771555
MCM6,0.000100,output/chromatin_dg1_bias_correction//MCM6_kap...,gene,0.0076,500,7,120710,121210
DSE1,0.000144,output/chromatin_dg1_bias_correction//DSE1_kap...,gene,0.0076,500,5,408854,409354
CLB2,0.000144,output/chromatin_dg1_bias_correction//CLB2_kap...,gene,0.0076,500,16,771055,771555


In [16]:
meta_df.loc['DSE1']

,path,group,gamma,window,chrom,start,end
kappa,,,,,,,
0.000100,output/chromatin_dg1_bias_correction//DSE1_kap...,gene,0.0076,500,5,408854,409354
0.000144,output/chromatin_dg1_bias_correction//DSE1_kap...,gene,0.0076,500,5,408854,409354
0.000207,output/chromatin_dg1_bias_correction//DSE1_kap...,gene,0.0076,500,5,408854,409354
0.000298,output/chromatin_dg1_bias_correction//DSE1_kap...,gene,0.0076,500,5,408854,409354
0.000428,output/chromatin_dg1_bias_correction//DSE1_kap...,gene,0.0076,500,5,408854,409354
0.000616,output/chromatin_dg1_bias_correction//DSE1_kap...,gene,0.0076,500,5,408854,409354
0.000886,output/chromatin_dg1_bias_correction//DSE1_kap...,gene,0.0076,500,5,408854,409354
0.001274,output/chromatin_dg1_bias_correction//DSE1_kap...,gene,0.0076,500,5,408854,409354
0.001833,output/chromatin_dg1_bias_correction//DSE1_kap...,gene,0.0076,500,5,408854,409354
